In [0]:
%run ./01_setup_volumen

In [0]:
from pydantic import BaseModel, Field
from typing import Literal

from datetime import datetime, timezone
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.json as pajson
import pyarrow.parquet as pq

class OrderContract(BaseModel):
    order_id: int
    customer_id: int
    product_id: int
    quantity: int = Field(gt=0)
    order_ts: datetime
    status: Literal["paid", "cancelled", "pending"]

In [0]:
sample_size = 1000
validated = 0

with open(settings.raw / "orders.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= sample_size:
            break
        OrderContract.model_validate_json(line)
        validated += 1
print(f"Muestra validada con Pydantic: {validated}")

In [0]:
def add_ingestion_metadata(table: pa.Table, dataset_name: str, source_file: str) -> pa.Table:
    n = table.num_rows
    ingestion_ts = datetime.now(timezone.utc).isoformat()
    table = table.append_column("dataset_name", pa.array([dataset_name] * n))
    table = table.append_column("source_file", pa.array([source_file] * n))
    table = table.append_column("batch_id", pa.array([settings.batch_id] * n))
    table = table.append_column("ingestion_ts", pa.array([ingestion_ts] * n))
    table = table.append_column("run_date", pa.array([settings.run_date] * n))
    return table


def write_bronze(table: pa.Table, dataset_name: str) -> Path:
    out_dir = settings.bronze / dataset_name / f"run_date={settings.run_date}"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / "part-000.parquet"
    pq.write_table(table, out_file, compression="snappy")
    return out_file

In [0]:
customers_table = pacsv.read_csv(settings.raw / "customers.csv")

customers_table = add_ingestion_metadata(customers_table, "customers", "customers.csv")

write_bronze(customers_table, "customers")

In [0]:
import polars as pl

# Crea un DataFrame de Polars desde la tabla Arrow
df_polars = pl.from_arrow(customers_table)

df_polars.head()

In [0]:
products_table = pq.read_table(settings.raw / "products.parquet")
products_table = add_ingestion_metadata(products_table, "products", "products.parquet")
write_bronze(products_table, "products")

In [0]:
import polars as pl

# Crea un DataFrame de Polars desde la tabla Arrow
df_polars = pl.from_arrow(products_table)

df_polars.head()

In [0]:
orders_table = pajson.read_json(settings.raw / "orders.jsonl")
orders_table = add_ingestion_metadata(orders_table, "orders", "orders.jsonl")
write_bronze(orders_table, "orders")

In [0]:
import polars as pl

# Crea un DataFrame de Polars desde la tabla Arrow
df_polars = pl.from_arrow(orders_table)

df_polars.head()

In [0]:
payments_table = pacsv.read_csv(settings.raw / "payments.csv")
payments_table = add_ingestion_metadata(payments_table, "payments", "payments.csv")
write_bronze(payments_table, "payments")

In [0]:
import polars as pl

# Crea un DataFrame de Polars desde la tabla Arrow
df_polars = pl.from_arrow(payments_table)

df_polars.head()